# LeetCode #107 — Binary Tree Level Order Traversal II: các tầng từ dưới lên

**Mức độ:** Medium · **Chủ đề:** Tree, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def levelOrderBottom(self, root: Optional[TreeNode]) -> List[List[int]]:
```

Bài này dùng lại khung BFS theo tầng của #102, chỉ đổi **thứ tự các tầng** trong kết quả. Nhưng đây là dịp tốt để luyện ba cách khác nhau và hiểu **chi phí** của mỗi cách "đảo".

Mục tiêu của notebook:

1. Ôn khung BFS theo tầng (`for _ in range(len(q))`).
2. Ba cách sắp tầng từ dưới lên: **đảo cuối**, **`appendleft` vào `deque`**, **DFS ghi theo độ sâu rồi đảo**.
3. Phân biệt "đảo thứ tự các tầng" với "đảo thứ tự trong mỗi tầng" (thứ tự trong tầng vẫn trái → phải).
4. Vì sao `list.insert(0, …)` là lựa chọn kém.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, trả về các giá trị theo **thứ tự tầng từ dưới lên**: tầng sâu nhất trước, gốc sau cùng. **Trong mỗi tầng** vẫn đọc từ trái sang phải.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2000]`.
- `-1000 <= Node.val <= 1000`

---

## Ví dụ

```text
    3
   / \
  9  20          ->  [[15, 7], [9, 20], [3]]
    /  \
   15   7

root = [1]  ->  [[1]]
root = []   ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Hàng đợi (queue) và `collections.deque`
3. Duyệt theo chiều rộng (BFS — Breadth-First Search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Hàng đợi (queue) và `collections.deque`

### Hàng đợi là gì?

Hàng đợi là cấu trúc **vào trước, ra trước** (FIFO — First In, First Out), giống xếp hàng mua vé: người đến trước được phục vụ trước. Thêm ở **cuối**, lấy ra ở **đầu**.

```text
enqueue 1, 2, 3:     đầu → [1, 2, 3] ← cuối
dequeue → 1:         đầu → [2, 3]
enqueue 4:           đầu → [2, 3, 4]
```

### Vì sao không dùng `list.pop(0)`?

`list` lưu liền nhau; bỏ phần tử đầu buộc **dời** mọi phần tử còn lại sang trái → `O(n)`. `collections.deque` (hàng đợi hai đầu) thêm/bỏ ở **cả hai đầu** trong `O(1)`.

| Thao tác | `deque` | `list` |
|---|---|---|
| thêm cuối `append` | `O(1)` | `O(1)` |
| bỏ đầu `popleft` / `pop(0)` | `O(1)` | `O(n)` |
| thêm đầu `appendleft` / `insert(0, x)` | `O(1)` | `O(n)` |

In [6]:
from collections import deque
import time as _time_queue

q = deque()
for x in [1, 2, 3]:
    q.append(x)                  # vào cuối
print("hàng đợi:", q)
print("popleft:", q.popleft(), "->", q)   # ra đầu
q.appendleft(0)
print("appendleft(0):", q, "| xem đầu:", q[0], "| xem cuối:", q[-1], "| len:", len(q))

_n = 50_000
_t0 = _time_queue.perf_counter()
_l = list(range(_n))
while _l:
    _l.pop(0)
_t1 = _time_queue.perf_counter()
_d = deque(range(_n))
while _d:
    _d.popleft()
_t2 = _time_queue.perf_counter()
print(f"bỏ đầu {_n} lần: list.pop(0) {_t1 - _t0:.3f}s | deque.popleft {_t2 - _t1:.3f}s")

hàng đợi: deque([1, 2, 3])
popleft: 1 -> deque([2, 3])
appendleft(0): deque([0, 2, 3]) | xem đầu: 0 | xem cuối: 3 | len: 3


bỏ đầu 50000 lần: list.pop(0) 0.874s | deque.popleft 0.030s


### Khi nào dùng hàng đợi?

- **BFS (duyệt theo chiều rộng)**: xử lý các đỉnh theo thứ tự được phát hiện — gần trước, xa sau.
- Mô phỏng hàng chờ, xử lý tác vụ theo lượt.
- `deque` cũng làm được ngăn xếp (`append` + `pop`) và **cửa sổ trượt** (bỏ phần tử cũ ở đầu).

Lỗi hay gặp: `q.pop()` (bỏ **cuối**) thay vì `q.popleft()` → vô tình biến hàng đợi thành ngăn xếp, BFS thành DFS.

## 2.3 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [7]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [8]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [9]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.4 Áp dụng vào bài này

**BFS theo tầng** trên cây như #102, rồi trả các tầng theo thứ tự **từ dưới lên** (đảo danh sách các tầng, hoặc `appendleft` từng tầng vào một `deque`).

# 3. Nền tảng

In [10]:
from typing import List, Optional
from collections import deque
import random, timeit


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

## Khung BFS theo tầng (#102)

Đầu mỗi vòng `while`, hàng đợi chứa **đúng một tầng**. Chụp `len(q)` rồi lấy ra đúng chừng ấy nút.

In [11]:
def level_order(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.append(level)
    return result

print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


# 4. Cách 1: BFS bình thường rồi đảo danh sách tầng

Đảo **danh sách các tầng** (không đảo nội dung từng tầng). `result[::-1]` hoặc `result.reverse()` đều `O(số tầng)` — không đáng kể.

In [12]:
def bottom_reverse(root: Optional[TreeNode]) -> List[List[int]]:
    return level_order(root)[::-1]

print(bottom_reverse(build_tree([3, 9, 20, None, None, 15, 7])))

[[15, 7], [9, 20], [3]]


# 5. Cách 2: `appendleft` từng tầng vào một `deque`

Mỗi khi xong một tầng, đặt nó lên **đầu** kết quả. `deque.appendleft` là `O(1)`.

In [13]:
def bottom_deque(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = deque(), deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.appendleft(level)
    return list(result)

print(bottom_deque(build_tree([3, 9, 20, None, None, 15, 7])))

[[15, 7], [9, 20], [3]]


# 6. Mô phỏng

In [14]:
def trace_bottom(values: list) -> List[List[int]]:
    root = build_tree(values)
    result, q, depth = deque(), deque([root]) if root else deque(), 0
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.appendleft(level)
        print(f"Xong tầng {depth}: {level} -> appendleft -> result = {list(result)}")
        depth += 1
    return list(result)

_ = trace_bottom([3, 9, 20, None, None, 15, 7])

Xong tầng 0: [3] -> appendleft -> result = [[3]]
Xong tầng 1: [9, 20] -> appendleft -> result = [[9, 20], [3]]
Xong tầng 2: [15, 7] -> appendleft -> result = [[15, 7], [9, 20], [3]]


# 7. Cách 3: DFS ghi theo độ sâu

DFS (gốc → trái → phải) đưa giá trị vào `levels[depth]`; cuối cùng đảo. Thăm trái trước phải đảm bảo thứ tự trong tầng là trái → phải.

In [15]:
def bottom_dfs(root: Optional[TreeNode]) -> List[List[int]]:
    levels = []
    def dfs(node, depth):
        if not node:
            return
        if depth == len(levels):
            levels.append([])
        levels[depth].append(node.val)
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return levels[::-1]

print(bottom_dfs(build_tree([3, 9, 20, None, None, 15, 7])))

[[15, 7], [9, 20], [3]]


# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def levelOrderBottom(self, root: Optional[TreeNode]) -> List[List[int]]:
        if not root:
            return []
        result, q = deque(), deque([root])
        while q:
            level = []
            for _ in range(len(q)):
                node = q.popleft()
                level.append(node.val)
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
            result.appendleft(level)
        return list(result)

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


def random_tree(n):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(-9, 9), random_tree(k), random_tree(n - 1 - k))


sol = Solution()
check("Ví dụ 1", sol.levelOrderBottom(build_tree([3, 9, 20, None, None, 15, 7])), [[15, 7], [9, 20], [3]])
check("Ví dụ 2", sol.levelOrderBottom(build_tree([1])), [[1]])
check("Ví dụ 3", sol.levelOrderBottom(build_tree([])), [])
check("Lệch trái", sol.levelOrderBottom(build_tree([1, 2, None, 3])), [[3], [2], [1]])

random.seed(107)
for _ in range(3000):
    root = random_tree(random.randint(0, 15))
    expected = level_order(root)[::-1]
    assert sol.levelOrderBottom(root) == expected == bottom_dfs(root) == bottom_reverse(root)
print("3000 cây ngẫu nhiên: 4 cách khớp nhau")

Ví dụ 1        PASS  [[15, 7], [9, 20], [3]]
Ví dụ 2        PASS  [[1]]
Ví dụ 3        PASS  []
Lệch trái      PASS  [[3], [2], [1]]


3000 cây ngẫu nhiên: 4 cách khớp nhau


# 9. Vì sao đúng? Độ phức tạp?

BFS theo tầng cho các tầng từ trên xuống, mỗi tầng trái → phải (bất biến của #102). `appendleft` từng tầng (hoặc đảo cuối) đảo **thứ tự các tầng** mà không chạm vào nội dung mỗi tầng.

```text
Time:  O(n)
Space: O(w) cho hàng đợi (w = độ rộng lớn nhất) + O(n) cho output
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Đảo luôn nội dung mỗi tầng

In [18]:
def wrong_reverse_all(root: Optional[TreeNode]) -> List[List[int]]:
    flat = [v for level in level_order(root) for v in level][::-1]   # SAI: đảo cả trong tầng
    out, i = [], 0
    for level in level_order(root)[::-1]:
        out.append(flat[i:i + len(level)])
        i += len(level)
    return out

print("Sai :", wrong_reverse_all(build_tree([3, 9, 20, None, None, 15, 7])), "| Đúng:", sol.levelOrderBottom(build_tree([3, 9, 20, None, None, 15, 7])))

Sai : [[7, 15], [20, 9], [3]] | Đúng: [[15, 7], [9, 20], [3]]


Tầng `[15, 7]` thành `[7, 15]`, tầng `[9, 20]` thành `[20, 9]`. Đề chỉ đảo thứ tự **giữa** các tầng.

## Lỗi 2: `result.insert(0, level)` với list

Đúng kết quả, nhưng mỗi lần chèn đầu list tốn `O(số tầng hiện có)`. Với cây lệch (mỗi tầng một nút, 2000 tầng) tổng chi phí là `O(h²)`.

In [19]:
def bottom_insert0(root):
    if not root:
        return []
    result, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        result.insert(0, level)
    return result

chain = None
for v in range(20000):
    chain = TreeNode(v, chain)
t_insert = timeit.timeit(lambda: bottom_insert0(chain), number=1)
t_deque = timeit.timeit(lambda: sol.levelOrderBottom(chain), number=1)
print(f"Cây lệch 20000 tầng: insert(0) {t_insert:.3f}s | deque.appendleft {t_deque:.3f}s | kết quả giống nhau? {bottom_insert0(chain) == sol.levelOrderBottom(chain)}")

Cây lệch 20000 tầng: insert(0) 0.175s | deque.appendleft 0.016s | kết quả giống nhau? True


(Với 2000 nút như đề, chênh lệch nhỏ; nhưng nên biết vì sao `deque` tốt hơn.)

## Lỗi 3: DFS thăm phải trước trái

Thứ tự trong mỗi tầng bị đảo.

## Lỗi 4: Quên trường hợp `root = None`

`deque([None])` → lỗi `AttributeError` ở vòng đầu.

# 11. Mẹo nhận diện pattern

"Theo tầng, nhưng thứ tự khác" → giữ nguyên BFS, đổi cách **ghi**:

| Yêu cầu | Cách ghi |
|---|---|
| Tầng trên xuống (#102) | `result.append(level)` |
| Tầng dưới lên (#107) | `result.appendleft(level)` hoặc đảo cuối |
| Zigzag (#103) | `level.appendleft(val)` ở tầng lẻ |
| Nhìn từ phải (#199) | lấy `level[-1]` |
| Trung bình mỗi tầng (#637) | `sum(level) / len(level)` |

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Kết quả `levelOrderBottom` của cây `[1, 2, 3, 4, None, None, 5, None, 6]`.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_level_order_bottom(root: Optional[TreeNode]) -> List[List[int]]:
    if not root:
        return []
    result, q = deque(), deque([root])
    while q:
        level = []
        # TODO: xử lý đúng một tầng
        # TODO: đặt tầng lên đầu kết quả
    return list(result)


# assert exercise_2_level_order_bottom(build_tree([3, 9, 20, None, None, 15, 7])) == [[15, 7], [9, 20], [3]]
# assert exercise_2_level_order_bottom(build_tree([])) == []

## Bài 3 — Biến thể: Find Largest Value in Each Tree Row (LeetCode #515)

```python
largest_values([1, 3, 2, 5, 3, None, 9]) -> [1, 3, 9]
```

In [21]:
def exercise_3_largest_values(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_3_largest_values(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 3, 9]
# assert exercise_3_largest_values(build_tree([])) == []

## Bài 4 — Biến thể: Find Bottom Left Tree Value (LeetCode #513)

Giá trị nút **trái nhất ở tầng sâu nhất**. Mẹo: BFS mà thêm con **phải trước trái** — nút cuối cùng lấy ra khỏi hàng đợi chính là đáp án.

```python
bottom_left([2, 1, 3]) -> 1
bottom_left([1, 2, 3, 4, None, 5, 6, None, None, 7]) -> 7
```

In [22]:
def exercise_4_bottom_left(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_4_bottom_left(build_tree([2, 1, 3])) == 1
# assert exercise_4_bottom_left(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])) == 7

## Bài 5 — Mini challenge: Deepest Leaves Sum (LeetCode #1302) và Cousins (#993)

(a) Tổng các lá ở tầng **sâu nhất**.

(b) Hai nút `x`, `y` là **anh em họ** nếu cùng độ sâu nhưng **khác cha**. Kiểm tra.

```python
deepest_leaves_sum([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8]) -> 15
is_cousins([1, 2, 3, None, 4, None, 5], 5, 4) -> True
is_cousins([1, 2, 3, 4], 4, 3) -> False
```

In [23]:
def exercise_5_deepest_leaves_sum(root: Optional[TreeNode]) -> int:
    # TODO
    pass


def exercise_5_is_cousins(root: Optional[TreeNode], x: int, y: int) -> bool:
    # TODO
    pass


# assert exercise_5_deepest_leaves_sum(build_tree([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8])) == 15
# assert exercise_5_is_cousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4) is True
# assert exercise_5_is_cousins(build_tree([1, 2, 3, 4]), 4, 3) is False
# assert exercise_5_is_cousins(build_tree([1, 2, 3, None, 4]), 2, 3) is False

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_bottom` sau khi tự làm.

**Bài 3**
- Mức 1: khung BFS, ghi `max(level)`.

**Bài 4**
- Mức 1: `q = deque([root])`; lặp `node = q.popleft()`; thêm `node.right` rồi `node.left`; trả `node.val` cuối cùng.

**Bài 5**
- Mức 1 (a): tổng của tầng cuối trong `level_order`.
- Mức 1 (b): BFS lưu `(node, parent)`; trong một tầng, tìm cha của `x` và `y`.
- Mức 2 (b): cả hai cùng tầng và cha khác nhau → True; chỉ một trong hai ở tầng này → False.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_bottom([1, 2, 3, 4, None, None, 5, None, 6])

Xong tầng 0: [1] -> appendleft -> result = [[1]]
Xong tầng 1: [2, 3] -> appendleft -> result = [[2, 3], [1]]
Xong tầng 2: [4, 5] -> appendleft -> result = [[4, 5], [2, 3], [1]]
Xong tầng 3: [6] -> appendleft -> result = [[6], [4, 5], [2, 3], [1]]


## Đáp án Bài 2

In [25]:
def answer_2_level_order_bottom(root: Optional[TreeNode]) -> List[List[int]]:
    return bottom_deque(root)

assert answer_2_level_order_bottom(build_tree([3, 9, 20, None, None, 15, 7])) == [[15, 7], [9, 20], [3]]
assert answer_2_level_order_bottom(build_tree([])) == []
print("Bài 2 PASS (giống bottom_deque ở mục 5)")

Bài 2 PASS (giống bottom_deque ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_largest_values(root: Optional[TreeNode]) -> List[int]:
    return [max(level) for level in level_order(root)]

assert answer_3_largest_values(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 3, 9]
assert answer_3_largest_values(build_tree([])) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_bottom_left(root: Optional[TreeNode]) -> int:
    q = deque([root])
    while q:
        node = q.popleft()
        if node.right:
            q.append(node.right)
        if node.left:
            q.append(node.left)
    return node.val

assert answer_4_bottom_left(build_tree([2, 1, 3])) == 1
assert answer_4_bottom_left(build_tree([1, 2, 3, 4, None, 5, 6, None, None, 7])) == 7
for _ in range(2000):
    root = random_tree(random.randint(1, 15))
    assert answer_4_bottom_left(root) == level_order(root)[-1][0]
print("Bài 4 PASS")

Bài 4 PASS


Vì sao đúng? Duyệt "phải trước trái" thì trong mỗi tầng nút **trái nhất** được lấy ra **cuối cùng**; và tầng sâu nhất được xử lý sau cùng. Nên nút cuối cùng là nút trái nhất của tầng sâu nhất.

## Đáp án Bài 5

In [28]:
def answer_5_deepest_leaves_sum(root: Optional[TreeNode]) -> int:
    levels = level_order(root)
    return sum(levels[-1]) if levels else 0


def answer_5_is_cousins(root: Optional[TreeNode], x: int, y: int) -> bool:
    q = deque([(root, None)])
    while q:
        parent_of = {}
        for _ in range(len(q)):
            node, parent = q.popleft()
            if node.val in (x, y):
                parent_of[node.val] = parent
            for child in (node.left, node.right):
                if child:
                    q.append((child, node))
        if len(parent_of) == 2:
            return parent_of[x] is not parent_of[y]
        if len(parent_of) == 1:
            return False
    return False

assert answer_5_deepest_leaves_sum(build_tree([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8])) == 15
assert answer_5_is_cousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4) is True
assert answer_5_is_cousins(build_tree([1, 2, 3, 4]), 4, 3) is False
assert answer_5_is_cousins(build_tree([1, 2, 3, None, 4]), 2, 3) is False
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> BFS theo tầng như #102; mỗi khi xong một tầng, `appendleft` nó vào `deque` kết quả (hoặc đảo danh sách tầng ở cuối). Chỉ đảo thứ tự **giữa** các tầng, trong tầng vẫn trái → phải.

---

## Checklist trước khi nộp

- [ ] `root = None` → `[]`?
- [ ] Không đảo nội dung từng tầng?
- [ ] Dùng `deque.appendleft` (hoặc đảo cuối), không `list.insert(0, …)`?
- [ ] Thêm con trái trước phải?